# Entrega 1 — desenho e configuração científica

**TCC: o que preservar os GEOs pode acrescentar ao Marketing Mix Modeling?**

Esta execução prepara e audita o painel e a especificação. Não estima a posterior nem resultados de marketing. A EDA já encontrou informação geotemporal, mas isso não comprova ganho causal ou preditivo. Vamos seguir uma sequência: origem → contrato → causalidade e especificação → priors → holdout → próximos experimentos → prontidão.

O guia completo é `docs/01_DESENHO_CIENTIFICO_DIDATICO.md`. O dataset é sintético, sem empresa, plataforma ou moeda identificadas. Não arredondamos conversões decimais. O objetivo desta sessão é verificar que a pergunta científica pode ser traduzida em entradas e decisões rastreáveis.

## 1. Preparação do ambiente

Em sessão Colab limpa, a próxima célula baixa **somente o repositório autorizado**. `REF` pode ser substituído pelo SHA da entrega para reprodução congelada; `main` serve para abrir a versão corrente e o SHA efetivo é impresso. A fonte dos dados é sempre um commit imutável com hash verificado. A instalação usa o lock do ambiente validado em Python 3.12. Se Colab pedir reinício, reinicie e execute novamente a partir desta célula. Localmente, o ambiente deve ter sido instalado pelos comandos do README; esta célula não instala nada fora do Colab.

In [1]:
import os, sys, subprocess
from pathlib import Path
REPOSITORY = 'https://github.com/thiagossilva17/MMM_Google_Meridian_ModelagemEtapa1.git'
REF = 'main'  # Para repetir a entrega, substitua pelo SHA informado no histórico.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    root = Path('/content/MMM_Google_Meridian_ModelagemEtapa1')
    if not root.exists():
        subprocess.run(['git', 'clone', REPOSITORY, str(root)], check=True)
    remote = subprocess.check_output(['git','-C',str(root),'remote','get-url','origin'],text=True).strip()
    assert remote == REPOSITORY, 'Remoto não autorizado'
    if REF != 'main':
        subprocess.run(['git','-C',str(root),'checkout','--detach',REF],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements-lock.txt'],check=True)
else:
    root = Path.cwd()
    if root.name == 'notebooks':
        root = root.parent
    assert (root/'configs/baseline_geo.yaml').exists(), 'Abra o notebook dentro do repositório'
    os.chdir(root)
sys.path.insert(0,str(root))
print('Python:',sys.version)
print('Diretório:',root)
head = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True)
print('Commit:',head.stdout.strip() if head.returncode == 0 else 'workspace local anterior à publicação')

Python: 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]
Diretório: /workspace/scratch/795adfbb054a/project
Commit: workspace local anterior à publicação


## 2. A origem vem antes dos números

A base oficial tem 40 mercados × 156 semanas. Não precisamos refazer a EDA; precisamos impedir que outra base entre silenciosamente. `verify_provenance` confere SHA-256, tamanho e Git blob. O CSV bruto é imutável. A cópia analítica só perde a coluna de índice após sua sequência ser validada.

O snapshot de dados e a release da API são commits distintos. O guia histórico não contém o hash da sua cópia do CSV: sua igualdade byte a byte continua não comprovada, embora estrutura e indicadores selecionados sejam comparáveis.

In [2]:
import json
from src.data_source import verify_provenance
from src.data_contract import load_panel, audit, structural_evidence
print('SHA-256 confirmado:',verify_provenance())
df = load_panel()
display(df.head())
display(audit(df))
print(json.dumps(structural_evidence(df),indent=2,ensure_ascii=False))

SHA-256 confirmado: d9ee016f7cd21f5c90b50da10a794af91a372b69f881d3caa266edd41fdafbf6
{
  "rows": 6240,
  "columns_clean": 19,
  "geos": 40,
  "weeks": 156,
  "date_min": "2021-01-25",
  "date_max": "2024-01-15",
  "population_kpi_corr": 0.9693338567456896,
  "between_kpi_fraction": 0.6788822241343846,
  "channel2_zero_fraction": 0.6426282051282052,
  "channel3_zero_fraction": 0.027083333333333334,
  "promo_zero_fraction": 0.46826923076923077,
  "channel2_spend_share": 0.05503310917455205,
  "channel3_spend_share": 0.4002888360053137,
  "population_once": 21696753.310000002,
  "fractional_conversions_preserved": true
}


,geo,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,competitor_sales_control,sentiment_score_control,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,Promo,conversions,revenue_per_conversion,population
0,Geo0,2021-01-25,280668,0,0,470611,108010,-1.338765,0.115581,2058.0608,0.00000,0.00000,3667.3965,841.6044,97320,0.000000,1954576.8,0.020055,136670.94
1,Geo0,2021-02-01,366206,182108,19825,527702,252506,0.893645,0.944224,2685.2874,1755.74540,147.31808,4112.2974,1967.5044,201441,0.000000,2064249.6,0.020103,136670.94
2,Geo0,2021-02-08,197565,230170,0,393618,184061,-0.284549,-1.290579,1448.6895,2219.12230,0.00000,3067.4023,1434.1870,0,0.683819,2086382.8,0.019929,136670.94
3,Geo0,2021-02-15,140990,66643,0,326034,201729,-1.034740,-1.084514,1033.8406,642.52057,0.00000,2540.7310,1571.8545,0,1.289055,2826431.5,0.019987,136670.94
4,Geo0,2021-02-22,399116,164991,0,381982,153973,-0.319276,-0.017503,2926.6072,1590.71640,0.00000,2976.7249,1199.7440,0,0.227739,3551929.2,0.020000,136670.94


,variable,dtype,min,max,zeros,negatives,total,status
0,Channel0_impression,int64,0.000000,5.192032e+06,755,0,5.523420e+09,PASS
1,Channel1_impression,int64,0.000000,4.397540e+06,1742,0,3.248579e+09,PASS
2,Channel2_impression,int64,0.000000,5.610156e+06,4010,0,1.625551e+09,PASS
3,Channel3_impression,int64,0.000000,7.635147e+06,169,0,1.127450e+10,PASS
4,Channel4_impression,int64,0.000000,6.975542e+06,820,0,6.125671e+09,PASS
5,Channel0_spend,float64,0.000000,3.807173e+04,755,0,4.050171e+07,PASS
6,Channel1_spend,float64,0.000000,4.239770e+04,1742,0,3.132030e+07,PASS
7,Channel2_spend,float64,0.000000,4.168864e+04,4010,0,1.207935e+07,PASS
8,Channel3_spend,float64,0.000000,5.949948e+04,169,0,8.786036e+07,PASS
9,Channel4_spend,float64,0.000000,5.435281e+04,820,0,4.773068e+07,PASS


## 3. Qual pergunta o modelo tentará responder?

A contribuição incremental futura compara a resposta esperada com a mídia observada à resposta esperada quando um canal é zerado, mantendo os demais fatores definidos. Só observamos uma história. A outra depende de pressupostos.

**DAG hipotético:** GEO/população e tempo influenciam demanda e KPI; demanda orienta investimento, Promo e orgânico; investimento determina exposição; exposição, orgânico e Promo podem influenciar KPI. Os índices podem ser proxies de demanda. Alternativamente, sentimento ou orgânico podem ser consequência da mídia: ajustá-los poderia remover mediação. Nenhum papel foi descoberto pela correlação.

Pressupostos: consistência da intervenção (unknown), ausência de confusão condicional (unsupported), positividade (partially assessable), não interferência (unknown), estabilidade temporal (assumed). O guia detalha o DAG e o registro. Na próxima entrega, boa convergência não converterá esses pressupostos em fatos.

Investigaremos precisão GEO/nacional, estabilidade a priors, heterogeneidade, atividade esparsa de C2, extrapolação de C3, tempo e papel dos controles. Não exigiremos que GEO vença.

DAG conceitual, explicitamente hipotético (o texto acima descreve as mesmas setas):

```mermaid
flowchart TD
  G["GEO e população"] --> D["Demanda latente"]
  T["Tempo"] --> D
  D --> I["Seleção de investimento"]
  I --> M["Exposição paga"]
  D --> Y["KPI"]
  G --> Y
  T --> Y
  M --> Y
  D --> C["Índices observados"]
  C --> I
  C --> Y
  D --> P["Promo"]
  P --> Y
  P --> I
  D --> O["Orgânico"]
  O --> Y
```

## 4. Traduzir variáveis sem mudar seu significado

`conversions` é o KPI contínuo não monetário. Impressões e gasto são pares por canal; gasto não entra como um segundo preditor duplicando exposição. População entra uma vez por GEO. Os controles preservam valores negativos. Promo preserva intensidade e zeros. Não há reach/frequency.

A API executável requer `non_revenue`, apesar de uma docstring escrever `non-revenue`. A configuração recebe valores brutos; as medidas por mil habitantes da EDA não substituem os dados de entrada. A célula abaixo constrói o objeto oficial, sem treinar.

In [3]:
import yaml
from src.meridian_setup import build_input, inspect_input, build_spec, prior_inventory, initialize_structure
input_data = build_input(df)
print(json.dumps(inspect_input(input_data,df),indent=2))
cfg = yaml.safe_load(Path('configs/baseline_geo.yaml').read_text())
display(cfg)
reference_spec = build_spec(input_data)
print('Prior paga ativa:',reference_spec.effective_media_prior_type)

{
  "kpi": {
    "shape": [
      40,
      156
    ],
    "dims": [
      "geo",
      "time"
    ]
  },
  "population": {
    "shape": [
      40
    ],
    "dims": [
      "geo"
    ]
  },
  "media": {
    "shape": [
      40,
      156,
      5
    ],
    "dims": [
      "geo",
      "media_time",
      "media_channel"
    ]
  },
  "media_spend": {
    "shape": [
      40,
      156,
      5
    ],
    "dims": [
      "geo",
      "time",
      "media_channel"
    ]
  },
  "controls": {
    "shape": [
      40,
      156,
      2
    ],
    "dims": [
      "geo",
      "time",
      "control_variable"
    ]
  },
  "organic_media": {
    "shape": [
      40,
      156,
      1
    ],
    "dims": [
      "geo",
      "media_time",
      "organic_media_channel"
    ]
  },
  "non_media_treatments": {
    "shape": [
      40,
      156,
      1
    ],
    "dims": [
      "geo",
      "time",
      "non_media_channel"
    ]
  }
}
Prior paga ativa: contribution


/workspace/scratch/795adfbb054a/venv/lib/python3.12/site-packages/meridian/data/input_data.py:605: UserWarning: Consider setting custom priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
  warnings.warn(


{'version': '2.1.0',
 'source_commit': 'cb0be8fe454124aadb73222c579f081f8823edfd',
 'source_url': 'https://github.com/google/meridian/blob/cb0be8fe454124aadb73222c579f081f8823edfd/meridian/model/spec.py',
 'status': 'explicit_choice',
 'outcome': 'conversions',
 'kpi_type': 'non_revenue',
 'revenue_per_kpi': None,
 'revenue_policy': 'pending_decision',
 'scope': 'estrutura_sem_inferencia',
 'model_spec': {'media_prior_type': {'value': 'contribution',
   'status': 'explicit_choice',
   'reason': 'Evitar impor ROI monetário sem unidade validada; default Beta(1,99) exige revisão preditiva.'},
  'organic_media_prior_type': {'value': 'contribution',
   'status': 'verified_default',
   'reason': 'Default técnico, não conhecimento de negócio.'},
  'non_media_treatments_prior_type': {'value': 'contribution',
   'status': 'verified_default',
   'reason': 'Permite sinal positivo ou negativo na contribuição de Promo.'},
  'non_media_baseline_values': {'value': {'Promo': 0.0},
   'status': 'explic

## 5. Adstock, saturação, tempo e priors

Adstock representa memória: pesos geométricos até oito semanas. Hill representa saturação após essa acumulação; ec é meia saturação na escala interna e slope é fixo em 1. `log_normal` compartilha informação entre mercados impondo efeitos de mídia positivos. Uma escala residual comum é hipótese inicial.

`knots=None` significa 156 knots neste GEO, mas apenas um no nacional. AKS está desligado. Isso impede interpretar uma comparação futura de defaults como se só a geografia mudasse.

A prior paga escolhida é contribuição Beta(1,99), default com média 1% por canal: **não é neutra**, e sua adequação continua REVIEW. Orgânico também usa contribuição; Promo usa contribuição com sinal e baseline proposto zero. Não há prior baseada em ROI real ou lift.

O aviso de InputData sobre fallback de contribuição total é genérico. A especificação explícita paga usa contribuição por canal; campos ROI/R&F presentes no inventário não significam que estejam ativos. Aqui só inspecionamos distribuições, sem amostragem. Prior predictive será feito na Entrega 2, revendo KPI negativo, baseline, escala e saturação antes da posterior.

In [4]:
import pandas as pd
inventory = prior_inventory(reference_spec)
display(pd.DataFrame(inventory))

,parameter,distribution,parameters
0,knot_values,Normal,"{'loc': 0.0, 'scale': 5.0}"
1,tau_g_excl_baseline,Normal,"{'loc': 0.0, 'scale': 5.0}"
2,beta_m,HalfNormal,{'scale': 5.0}
3,beta_rf,HalfNormal,{'scale': 5.0}
4,beta_om,HalfNormal,{'scale': 5.0}
5,beta_orf,HalfNormal,{'scale': 5.0}
6,eta_m,HalfNormal,{'scale': 1.0}
7,eta_rf,HalfNormal,{'scale': 1.0}
8,eta_om,HalfNormal,{'scale': 1.0}
9,eta_orf,HalfNormal,{'scale': 1.0}


## 6. H-GEO: reservar informação sem escolher pelo resultado

O algoritmo usa apenas geo/time, seed 20261025 e seis GEOs por semana. Ele garante 15% do painel: 23 ou 24 semanas reservadas por GEO. A tabela completa é salva; a seed sozinha não substitui a máscara.

Somente o KPI dessas células sai da likelihood. Mídia histórica continua disponível para adstock. Não é um exercício de forecasting futuro. Balanceamento de células não garante equilíbrio de mídia: confira sobretudo atividade e gasto de Channel2.

In [5]:
from src.validation_design import make_holdout, coverage
validation = yaml.safe_load(Path('configs/validation_design.yaml').read_text())
h = validation['H-GEO']
split = make_holdout(df[['geo','time']],seed=h['seed'],per_week=h['per_week'])
by_geo, by_week, by_channel = coverage(df,split)
display(by_geo)
display(by_week.head())
display(by_channel.query("channel == 'Channel2'"))
holdout_spec = build_spec(input_data,split)
structure = initialize_structure(input_data,holdout_spec,split)
print('Estrutura inicializada sem inferência:',structure)

Estrutura inicializada sem inferência: {'initialization': 'PASS', 'n_knots': 156, 'resolved_holdout_count': 936}


,geo,sum,count,fraction
0,Geo0,23,156,0.147436
1,Geo1,24,156,0.153846
2,Geo10,23,156,0.147436
3,Geo11,23,156,0.147436
4,Geo12,23,156,0.147436
5,Geo13,24,156,0.153846
6,Geo14,23,156,0.147436
7,Geo15,23,156,0.147436
8,Geo16,24,156,0.153846
9,Geo17,24,156,0.153846


,time,sum,count,fraction
0,2021-01-25,6,40,0.15
1,2021-02-01,6,40,0.15
2,2021-02-08,6,40,0.15
3,2021-02-15,6,40,0.15
4,2021-02-22,6,40,0.15


,geo,channel,subset,n,active,inactive,active_fraction,impressions,spend
4,ALL,Channel2,train,5304,1891,3413,0.356523,1.378170e+09,1.024108e+07
5,ALL,Channel2,test,936,339,597,0.362179,2.473816e+08,1.838274e+06
14,Geo0,Channel2,train,133,48,85,0.360902,8.948100e+06,6.649265e+04
15,Geo0,Channel2,test,23,6,17,0.260870,1.092485e+06,8.118173e+03
24,Geo1,Channel2,train,132,49,83,0.371212,1.433781e+07,1.065432e+05
...,...,...,...,...,...,...,...,...,...
385,Geo7,Channel2,test,23,11,12,0.478261,1.151555e+07,8.557119e+04
394,Geo8,Channel2,train,133,43,90,0.323308,1.961748e+07,1.457760e+05
395,Geo8,Channel2,test,23,4,19,0.173913,1.449645e+06,1.077220e+04
404,Geo9,Channel2,train,133,36,97,0.270677,2.949965e+07,2.192097e+05


## 7. H-COMP e nacional: decisões que continuam abertas

H-GEO não é holdout nacional equivalente: a soma nacional pode incluir KPI de células reservadas. R² de células e de somas também são unidades diferentes.

H-COMP está PENDING_METHOD_DECISION. Reservar semanas internas iguais nos dois modelos é candidato, mas deixar todos os GEOs de uma semana fora prejudica seu knot. Antes de executar, definir knots com suporte no treino e auditar uso indireto do KPI em seleção, transformação e priors.

O plano nacional soma KPI, impressões e gasto por semana e população uma vez por GEO. Receita por conversão seria ponderada por conversões. Os dois índices e Promo exigem uma hipótese de agregação: média populacional se intensidade, outra regra se sua semântica exigir. Somar índices por conveniência não resolve isso. Nenhum MMM nacional será criado aqui.

In [6]:
print(json.dumps(validation['H-COMP'],indent=2,ensure_ascii=False))
registry = yaml.safe_load(Path('configs/experiment_registry.yaml').read_text())
assert all(e['status']=='planned' for e in registry['experiments'])
display(pd.DataFrame(registry['experiments'])[['id','stage','status','hypothesis','factor_changed','required_gate']])

{
  "status": "PENDING_METHOD_DECISION",
  "candidate": "Semanas internas não contíguas iguais nos dois modelos, mantendo extremos no treino.",
  "fraction_proposed": 0.1,
  "seed_proposed": 20261026,
  "shared_scale": "Soma nacional de conversões por semana.",
  "required_before_execution": [
    "Definir knots interpoláveis com suporte no treino, iguais em desenho controlado quando viável.",
    "Não usar o KPI das semanas de teste para seleção de knots ou tuning.",
    "Auditar escala/priors dependentes do KPI e eventual uso indireto do teste.",
    "Fixar agregação dos índices e Promo com justificativa semântica.",
    "Separar comparação operacional de defaults da comparação metodológica controlada."
  ],
  "prohibited": "Comparar R2 das células GEO com R2 de totais nacionais; usar H-GEO como teste nacional equivalente."
}


,id,stage,status,hypothesis,factor_changed,required_gate
0,G0,2,planned,Referência GEO tecnicamente saudável,baseline,sampling_healthy
1,G1,2,planned,Referência suporta células omitidas,holdout H-GEO,fit_reviewed
2,N0,4,planned,Agregado é referência viável,agregação nacional e restrição temporal necess...,sampling_healthy
3,P1,3,planned,Contribuição paga é sensível à concentração,"Beta(0.5,49.5) paga, mesma média 1%",inference_reviewed
4,P2,3,planned,Contribuição paga é sensível ao centro,"Beta(2,98) paga, concentração 100",inference_reviewed
5,P3,3,planned,Parametrização altera conclusões,ROI somente após validar unidades e equivalênc...,inference_reviewed
6,P4,3,planned,Orgânico depende da prior,"Beta(0.5,49.5) orgânica, mesma média",inference_reviewed
7,A1,3,planned,Memória curta suficiente,max_lag=4,inference_reviewed
8,A2,3,planned,Memória longa muda atribuição,max_lag=12,inference_reviewed
9,A3,3,planned,Forma de decaimento importa,adstock_decay_spec=binomial,inference_reviewed


## 8. Executar a auditoria completa e ler seu alcance

Os gates futuros são dados → especificação → amostragem saudável → ajuste revisado → inferência revisada → eventual elegibilidade de otimização. Nesta etapa só há verificação técnica dos dois primeiros. R-hat, ESS, PPC e métricas de ajuste não existem ainda.

A execução abaixo salva as auditorias, a máscara, priors configuradas e o manifesto. Os seis alertas locais de C2 devem continuar REVIEW. A primeira falha de API foi registrada e corrigida; não se ocultam erros de construção.

In [7]:
from src.run_stage1 import run
manifest = run()
print(json.dumps(manifest,indent=2,ensure_ascii=False))
display(json.loads(Path('outputs/tables/holdout_review.json').read_text()))

{
  "executed_at_utc": "2026-10-25T11:46:05.192613+00:00",
  "scope": "Entrega 1: validação e configuração apenas",
  "data_ready": "PASS",
  "spec_ready": "PASS",
  "scientific_readiness": "REVIEW",
  "holdout_status": "REVIEW",
  "holdout_train": 5304,
  "holdout_test": 936,
  "H_COMP": "PENDING_METHOD_DECISION",
  "hosted_colab": "NOT_TESTED_IN_HOSTED_COLAB",
  "posterior_sampling_executed": false,
  "prior_sampling_executed": false,
  "roi_estimated": false,
  "national_model_constructed": false,
  "source_sha256": "d9ee016f7cd21f5c90b50da10a794af91a372b69f881d3caa266edd41fdafbf6",
  "python": "3.12.14",
  "versions": {
    "google-meridian": "2.1.0",
    "numpy": "2.3.5",
    "pandas": "2.3.3",
    "tensorflow": "2.21.0",
    "tfp-nightly": "0.26.0.dev20260130",
    "jax": "0.12.0.1",
    "PyYAML": "6.0.3",
    "pytest": "9.1.1",
    "nbformat": "5.11.1",
    "nbclient": "0.11.0",
    "ipykernel": "7.4.0"
  },
  "artifact_sha256": {
    "outputs/tables/data_audit.csv": "f7deea34fb

/workspace/scratch/795adfbb054a/venv/lib/python3.12/site-packages/meridian/data/input_data.py:605: UserWarning: Consider setting custom priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
  warnings.warn(


{'status': 'REVIEW',
 'alerts': ['Geo13/Channel2: diferença de atividade >20pp',
  'Geo29/Channel2: fração de gasto teste 0.0482 fora de [0.05,0.30]',
  'Geo29/Channel2: diferença de atividade >20pp',
  'Geo32/Channel2: fração de gasto teste 0.0497 fora de [0.05,0.30]',
  'Geo9/Channel2: fração de gasto teste 0.3095 fora de [0.05,0.30]',
  'Geo9/Channel2: diferença de atividade >20pp'],
 'action': 'Preservar máscara; investigar desequilíbrios antes de interpretar desempenho local.'}

## 9. Testes e encerramento da etapa

Os testes introduzem duplicações, ausências, população variável e pares quebrados para conferir rejeição; verificam imutabilidade, máscara independente do KPI, reconciliação de cobertura e integração real com Meridian. A suíte não instala pacotes nem ajusta modelos.

Uma execução local não comprova a interface hospedada do Colab. O campo NOT_TESTED_IN_HOSTED_COLAB permanece até registrar uma execução real em sessão limpa com evidência revisada; não é atualizado automaticamente só por abrir este notebook.

In [8]:
result = subprocess.run([sys.executable,'-m','pytest','-q'],capture_output=True,text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('BLOCKED: testes falharam')
print('PASS técnico: fonte, painel, InputData, ModelSpec e máscara verificados.')
print('REVIEW científico: priors, suporte local, pressupostos e uso indireto de KPI reservado.')
print('H-COMP: PENDING_METHOD_DECISION')
print('Colab hospedado: NOT_TESTED_IN_HOSTED_COLAB no registro original; anexar evidência para encerrá-lo.')
print('Fim da Entrega 1. Nenhuma posterior, ROI, mROI ou otimização foi executada.')

....................                                                     [100%]
=============================== warnings summary ===============================
tests/test_stage1.py::test_meridian_input_spec_resolved_mask
  /workspace/scratch/795adfbb054a/venv/lib/python3.12/site-packages/meridian/data/input_data.py:605: UserWarning: Consider setting custom priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
20 passed, 1 warning in 10.63s

PASS técnico: fonte, painel, InputData, ModelSpec e máscara verificados.
REVIEW científico: priors, suporte local, pressupostos e uso indireto de KPI reservado.
H-COMP: PE